# 🕵️‍♂️ Forensic Intelligence Platform: Technical Walkthrough
### **Industrial Data Factory: Ingestion ➡️ NLP ➡️ dbt ➡️ ML ➡️ XAI**

This notebook demonstrates the end-to-end data science lifecycle of the **Forensic Intelligence Platform**. It simulates the extraction of SEC filing text, the calculation of forensic narrative metrics, and the multi-model ML approach for anomaly detection and market forecasting.

## 🛠️ Phase 1: Setup & Data Ingestion (The Bronze Layer)
We pull unstructured 10-K text from the SEC and structured financial ratios from Yahoo Finance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from textblob import TextBlob
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import shap
import xgboost as xgb

# Sample Forensic Batch (Simulating Ingest from SEC & Yahoo)
data = {
    'ticker': ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'TSLA', 'XOM', 'NVDA'],
    'sector': ['Info Tech', 'Info Tech', 'Info Tech', 'Cons Discretionary', 'Info Tech', 'Cons Discretionary', 'Energy', 'Info Tech'],
    'raw_10k_text': [
        "The company performed well this fiscal year with strong revenue growth and healthy margins.",
        "We are optimistic about the future of cloud computing and our market share.",
        "Despite some headwinds in advertising, our core search engine remains robust.",
        "Consumer spending was strong; however, logistic costs increased due to supply chain issues.",
        "We are pivoting heavily to AI and the metaverse to ensure long-term engagement.",
        "Production challenges led to delays, but our energy segment is growing rapidly.",
        "Oil prices were volatile; however, our exploration and production segments stayed profitable.",
        "The demand for high-performance GPUs for AI training is unprecedented."
    ],
    'short_ratio': [1.2, 0.8, 1.5, 2.1, 1.9, 8.5, 1.1, 1.4],
    'debt_to_equity': [2.8, 1.5, 0.4, 1.2, 0.2, 0.5, 0.8, 0.3]
}
bronze_df = pd.DataFrame(data)
bronze_df.head()

## 🧪 Phase 2: Forensic NLP (Engine A)
Management uses **Word Density** to obfuscate and **Defensive Tone** to hide risks. We calculate `sentiment_score` and `word_count`.

In [ ]:
def calculate_forensic_metrics(row):
    # Sentiment Tone (-1.0 to +1.0)
    sentiment = TextBlob(row['raw_10k_text']).sentiment.polarity
    # Complexity (Simulating word count extraction)
    word_cnt = len(row['raw_10k_text'].split())
    return pd.Series([sentiment, word_cnt])

bronze_df[['sentiment_score', 'word_count']] = bronze_df.apply(calculate_forensic_metrics, axis=1)
bronze_df['density_index'] = (bronze_df['word_count'] / 10).round(2) # Simulated Density Offset
bronze_df.head()

## 🔄 Phase 3: Cloud Transformation (The Gold Layer)
In the production pipeline, we use **dbt** to calculate **Benford's Law Deviation** and **Narrative Drift** using SQL window functions.

In [ ]:
# Simulating dbt output (Gold Table)
bronze_df['benfords_deviation'] = [0.12, 0.08, 0.15, 0.22, 0.18, 0.85, 0.11, 0.14] # Critical Deviation in TSLA
bronze_df['narrative_drift'] = [0.02, 0.05, 0.08, 0.12, 0.15, 0.55, 0.04, 0.06] # Heavy obfuscation drift in TSLA

gold_df = bronze_df.copy()
gold_df.drop(columns=['raw_10k_text'], inplace=True)
gold_df

## 🤖 Phase 4: Machine Learning (Anomaly Detection)
We use **Isolation Forest** to find structural outliers in the cross-domain dataset.

In [ ]:
features = ['sentiment_score', 'word_count', 'short_ratio', 'debt_to_equity', 'benfords_deviation', 'narrative_drift']
X = gold_df[features].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = IsolationForest(contamination=0.1, random_state=42)
gold_df['anomaly_score'] = model.fit_predict(X_scaled)

# Scale to 0-100% Probability
prob_scaler = MinMaxScaler(feature_range=(0, 100))
gold_df['risk_probability'] = prob_scaler.fit_transform(-model.decision_function(X_scaled).reshape(-1,1)).round(1)

gold_df[['ticker', 'risk_probability', 'anomaly_score']]

## 🔍 Phase 5: Explainable AI (SHAP Matrix)
We decompose the **why** behind the anomaly for TSLA.

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_scaled)

# Visualizing Anomaly Drivers for the first critical entry (Simulated index 5: TSLA)
shap.initjs()
shap.force_plot(explainer.expected_value, shap_values[5,:], X.iloc[5,:], feature_names=features)

## 🔮 Phase 6: Predictive Forecasting (XGBoost)
Forecasting future short-selling engagement based on present forensic markers.

In [ ]:
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100)
# Training on the batch (In production, trained on 100+ tickers)
xgb_model.fit(X_scaled, gold_df['short_ratio'])

gold_df['expected_short_spike'] = xgb_model.predict(X_scaled).round(2)
gold_df[['ticker', 'short_ratio', 'expected_short_spike']]

## 🏁 Conclusion
The **Forensic Intelligence Platform** combines unstructured NLP narrative tone with quantitative accounting rigor and market pressure. By triangulating these signals and explaining them via SHAP, we provide auditors with immediate, defensible high-risk leads.